# AzLegalRAG - Colab Demo

Azerbaijani Legal Q&A using RAG with gte-Qwen2-7B embeddings (77.18% NDCG).

**Requirements**: L4 GPU (24GB VRAM)

Go to: Runtime > Change runtime type > L4 GPU

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers accelerate sentence-transformers langchain langchain-community langchain-core langchain-text-splitters chromadb datasets tqdm huggingface_hub

## 2. HuggingFace Login (Required for Gated Dataset)

The `allmalab/eqanun` dataset is gated. You need to:
1. Create a token at https://huggingface.co/settings/tokens
2. Accept access at https://huggingface.co/datasets/allmalab/eqanun
3. Run the cell below and paste your token

In [ ]:
from huggingface_hub import login
login()

## 3. Clone Repository

In [ ]:
!git clone https://github.com/StartZer0/AzLegalRAG.git
%cd AzLegalRAG

## 4. Check GPU

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 5. Ingest Documents (Run Once - Takes ~60 min)

This uses gte-Qwen2-7B-instruct (14GB) for embeddings.

In [ ]:
import sys
sys.path.insert(0, './src')

from ingest import load_eqanun, chunk_documents

# Load dataset
dataset = load_eqanun()
print(f"Loaded {len(dataset)} documents")

# Chunk documents
chunks = chunk_documents(dataset)
print(f"Created {len(chunks)} chunks")

In [ ]:
# Create vectorstore with gte-Qwen2-7B embeddings
# This loads the 14GB embedding model and processes all chunks
from embed import create_vectorstore

vectorstore = create_vectorstore(chunks)
print("Vectorstore created!")

## 6. Test Search (Without LLM)

This loads embedding model to encode query.

In [ ]:
from embed import load_vectorstore, unload_embeddings
from retrieve import semantic_search

vs = load_vectorstore()
results = semantic_search(vs, "Emek muqavilesi nedir?", k=3)

for i, doc in enumerate(results, 1):
    print(f"\n[{i}] Source: {doc.metadata['source']}")
    print(doc.page_content[:300] + "...")

## 7. Unload Embeddings, Load LLM

Sequential loading: unload embedding model (14GB) to make room for Mistral-7B (14GB).

In [ ]:
# Unload embedding model first
from embed import clear_gpu_memory
clear_gpu_memory()

# Now load LLM
from generate import get_llm, create_rag_chain

# Reload vectorstore WITHOUT loading embeddings (we just need the stored vectors)
from embed import load_vectorstore
vs = load_vectorstore(load_embeddings=True)  # Still need for query encoding

# Load LLM
llm = get_llm()

# Create RAG chain
chain = create_rag_chain(vs, llm)
print("RAG chain ready!")

## 8. Test RAG Q&A

In [ ]:
question = "Emek muqavilesi nedir?"
result = chain({"query": question})

print("=" * 50)
print(f"Question: {question}")
print("=" * 50)
print(f"\nAnswer:\n{result['result']}")
print("\n" + "=" * 50)
print("Sources:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}")

In [ ]:
# Try more questions
questions = [
    "Mehkeme qerari nece shekillendirilir?",
    "Nikah muqavilesi ucun ne teleb olunur?",
    "Vergiler hansi novlere bolunur?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    result = chain({"query": q})
    print(f"A: {result['result'][:500]}...")

## 9. Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/AzLegalRAG
!cp -r ./vectorstore /content/drive/MyDrive/AzLegalRAG/
print("Vectorstore saved to Google Drive!")